# Animation Factory — Colab GPU Render

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/l2yao/animation-factory/blob/main/notebooks/colab_render.ipynb)

Run the factory on free Colab T4/A100 GPU. Factory targets Blender 5.x — Factory is installable `pip` package + `bpy` pipeline.

1. Runtime → Change runtime type → **T4 GPU** (or A100)
2. Run all cells
3. Edit `YOUR_TEXT` in Cell 3 for any future film — reusable, not one-off


## 0. Mount Drive (optional, for persistence)

In [ ]:
from google.colab import drive
import os
USE_DRIVE = True  # set False to keep in /content ephemeral
if USE_DRIVE and not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print('Drive mounted')
else:
    print('Drive mount skipped or already mounted')

## 1. Clone factory + install Blender on Colab

In [ ]:
!test -d /content/animation-factory || git clone https://github.com/l2yao/animation-factory.git /content/animation-factory
%cd /content/animation-factory
!git pull --rebase 2>&1 | tail -n5
!pip install -q -e . 2>&1 | tail -n5
!factory --help

In [ ]:
# Install Blender 5.x to /opt/blender — ~300MB, cached after first run
!bash colab/setup.sh 2>&1 | tail -n20
!/opt/blender/blender --version 2>&1 | head -n5
!ffmpeg -version 2>&1 | head -n1
!nvidia-smi 2>&1 | head -n15

## 2. Create a new film from ANY text (reusable factory)
Edit `YOUR_TEXT` — this is the only cell you change for future films.

In [ ]:
YOUR_TEXT = """A lost robot finds friendship in a Pixar city, where glowing streets and kind strangers help it discover that being different is what makes it special."""
FILM_NAME = "robot-city-colab-001"  # change for each new film; keeps history in films/
DURATION = 60  # seconds target (30-120)
PRESET = "pixar"  # style preset
RENDER_PRESET = "render_colab"  # colab GPU tier (EEVEE 1080p, or edit presets/render_colab.yaml for Cycles)

# Write text to temp file for CLI (avoids shell quoting issues)
with open("/tmp/input.txt", "w", encoding="utf-8") as f:
    f.write(YOUR_TEXT)

!factory new-film --name $FILM_NAME --input /tmp/input.txt --preset $PRESET --duration $DURATION --force 2>&1
!cat films/$FILM_NAME/config.yaml
!cat films/$FILM_NAME/input.txt | head -n5

In [ ]:
# Parse → expand short prompt into Pixar beats (5 shots for your robot story)
!factory generate --film $FILM_NAME --step parse 2>&1
!cat films/$FILM_NAME/script.md
import json, pathlib
print(json.dumps(json.loads(pathlib.Path(f'films/{FILM_NAME}/shots.json').read_text())['shots'][:2], indent=2))

## 3. Build + Animate (generates Blender Python scripts)

In [ ]:
!factory generate --film $FILM_NAME --step build 2>&1
!factory generate --film $FILM_NAME --step animate 2>&1
!ls -lh films/$FILM_NAME/scripts/
!head -n40 films/$FILM_NAME/scripts/build_scene.py

## 4. Render on Colab GPU (Blender headless)
Render headless on Colab GPU. Uses Blender at `/opt/blender/blender -b`.


In [ ]:
# Option A: via factory (wraps blender -b)
!factory generate --film $FILM_NAME --step render 2>&1 | tail -n50

# Option B: direct Blender (if you want to debug):
# !/opt/blender/blender -b films/$FILM_NAME/scenes/shot_animated.blend -a 2>&1 | tail -n50
# If shot_animated.blend missing, build first:
# !/opt/blender/blender -b -P films/$FILM_NAME/scripts/build_scene.py 2>&1 | tail -n20
# !/opt/blender/blender -b -P films/$FILM_NAME/scripts/animate.py 2>&1 | tail -n20

!ls -lh films/$FILM_NAME/render/frames/ 2>&1 | head -n20
!ls films/$FILM_NAME/render/frames/ | wc -l

In [ ]:
# Quick preview — first frame
from IPython.display import Image, display
import glob
frames = sorted(glob.glob(f"films/{FILM_NAME}/render/frames/*.png"))
if frames:
    display(Image(filename=frames[0], width=480))
    print(f"Rendered {len(frames)} frames")
else:
    print("No frames yet — check render logs above")

## 5. Audio + Multi-aspect Export (YouTube / TikTok / IG)
Master 16:9 → 9:16 and 1:1 via `ffmpeg crop` (no re-render).

In [ ]:
!factory generate --film $FILM_NAME --step audio 2>&1 | tail -n20
!factory generate --film $FILM_NAME --step export 2>&1 | tail -n30
!ls -lh films/$FILM_NAME/final/ 2>&1
!ls -lh films/$FILM_NAME/audio/ 2>&1

In [ ]:
# Play result in Colab
from IPython.display import Video
import pathlib
for asp in ["16x9", "9x16", "1x1"]:
    p = pathlib.Path(f"films/{FILM_NAME}/final/{asp}.mp4")
    if p.exists():
        print(f"{asp}: {p.stat().st_size/1024/1024:.1f} MB")
        display(Video(str(p), width=360, embed=True))
    else:
        print(f"{asp}.mp4 not found")

## 6. Download / Save to Drive (reusable)


In [ ]:
import shutil, pathlib
if USE_DRIVE:
    dst = pathlib.Path(f"/content/drive/MyDrive/animation-factory/films/{FILM_NAME}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(f"films/{FILM_NAME}", dst)
    print(f"Saved to Drive: {dst}")
    # also save final videos separately
    for mp4 in pathlib.Path(f"films/{FILM_NAME}/final").glob("*.mp4"):
        print(f"  {mp4.name}: {mp4.stat().st_size/1e6:.1f} MB")
else:
    print("USE_DRIVE=False — files in /content will be lost on session end. Set True and re-run.")

# Download to local (Colab file browser → right-click, or use files.download)
from google.colab import files
# files.download(f"films/{FILM_NAME}/final/16x9.mp4")  # uncomment to auto-download

## 7. Reuse for next film (no code changes)
Just change `YOUR_TEXT` + `FILM_NAME` in Cell 2 and Re-run from Cell 2 onward. Factory keeps `presets/`, `assets_library/` shared.

| Task | Command |
|------|---------|
| List films | `!factory list` |
| Info | `!factory info --film robot-city-colab-001` |
| New aspect only | `!factory generate --film X --step export --aspect 9:16` |
| Cycles instead of EEVEE | edit `presets/render_colab.yaml` → `engine: CYCLES` then re-render |